# Held-out EM detection

Reproduces the main detection table and appendix regressor grid from calibration and held-out artifacts. Labels are stored Betley grades; fitting and evaluation run locally without model weights or judge calls.


In [1]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src/reproduction/reports.py").exists())
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display
from src.reproduction.reports import run_analysis, show_report

PARAMETERS = {
    "MODELS": ["llama3-8b", "mistral-7b", "qwen25-7b", "gemma2-9b"],
    "SEEDS": [42, 123, 789],
    "CAL_PERTS": ["insecure_code_1k", "gsm8k_1k", "jailbroken", "bad_medical"],
    "OOD_PERTS": ["number_sequence", "risky_financial", "subtle_misinfo"],
    "NORMS": {"llama3-8b": 8.5, "mistral-7b": 4.6875, "qwen25-7b": 66.5, "gemma2-9b": 372.0},
    "EM_THRESH": 0.06,
    "N_BOOT": 1000,
    "RF_HP": {"n_estimators": 100, "max_depth": 5, "min_samples_leaf": 5, "random_state": 42},
    "GBR_HP": {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1, "random_state": 42},
    "RIDGE_KW": {"alpha": 1.0},
}
display(pd.DataFrame([{"Parameter": key, "Value": str(value)} for key, value in PARAMETERS.items()]))


,Parameter,Value
0,MODELS,"['llama3-8b', 'mistral-7b', 'qwen25-7b', 'gemm..."
1,SEEDS,"[42, 123, 789]"
2,CAL_PERTS,"['insecure_code_1k', 'gsm8k_1k', 'jailbroken',..."
3,OOD_PERTS,"['number_sequence', 'risky_financial', 'subtle..."
4,NORMS,"{'llama3-8b': 8.5, 'mistral-7b': 4.6875, 'qwen..."
5,EM_THRESH,0.06
6,N_BOOT,1000
7,RF_HP,"{'n_estimators': 100, 'max_depth': 5, 'min_sam..."
8,GBR_HP,"{'n_estimators': 100, 'max_depth': 3, 'learnin..."
9,RIDGE_KW,{'alpha': 1.0}


## Persona baseline

Refit all three regressors from saved signed Persona checkpoint shifts. Vector extraction and layer selection are upstream experiments, not rerun here.


In [2]:
persona = run_analysis("persona_score", PARAMETERS)
recomputed = json.loads((persona["output"] / "chen_evil_persona_vector_baseline.json").read_text())
reference = json.loads((ROOT / "results/staging/chen_persona_vector/full/chen_evil_persona_vector_baseline.json").read_text())
assert recomputed["results"] == reference["results"]
assert recomputed["selected_regressor"] == reference["selected_regressor"]
display(pd.DataFrame([{"Regressor": name, **result["ood_pooled"]} for name, result in recomputed["results"].items()]).drop(columns=["ci95_run_bootstrap"]))


,Regressor,accuracy,balanced_accuracy,fnr,fpr,tp,fp,fn,tn,auroc
0,Ridge,0.846154,0.854077,0.036866,0.254980,209,64,8,187,0.939450
1,GBR,0.878205,0.875218,0.165899,0.083665,181,21,36,230,0.905668
2,RF,0.886752,0.884435,0.147465,0.083665,185,21,32,230,0.932354


## Main and full detection tables


In [3]:
report = run_analysis("headline", PARAMETERS)
checks = show_report(report, ["tab_headline_detection_main.tex","tab_headline_detection.tex"])
if not checks.empty:
    display(checks)


headline detection


,Features,ML,Acc (%),FNR % [95% CI],FPR % [95% CI],FN,FP,AUROC
0,Our 7D,RF,98.1,"1.8 [0.4, 3.6]","2.0 [0.4, 4.4]",4,5,0.989
1,\textit{Our 7D},\textit{GBR},\textit{92.7},"\textit{2.3 [0.6, 4.1]}","\textit{11.6 [3.2, 21.7]}",\textit{5},\textit{29},\textit{0.983}
2,Our 7D,Ridge,81.0,"16.6 [5.8, 29.6]","21.1 [7.7, 39.1]",36,53,0.936
3,|PC1| only,RF,88.7,"18.9 [7.8, 32.1]","4.8 [1.6, 9.2]",41,12,0.939
4,|PC1| only,GBR,89.1,"19.8 [8.2, 34.4]","3.2 [0.5, 7.3]",43,8,0.902
5,|PC1| only,Ridge,79.7,"19.4 [8.6, 32.0]","21.1 [7.4, 38.6]",42,53,0.913
6,Persona,Ridge,84.6,"3.7 [0.0, 9.9]","25.5 [11.8, 42.0]",8,64,0.939
7,Persona,RF,88.7,"14.7 [3.1, 28.6]","8.4 [3.5, 14.5]",32,21,0.932
8,Persona,GBR,87.8,"16.6 [5.0, 31.5]","8.4 [3.5, 14.6]",36,21,0.906
9,\|Δ \bar{h}\|_2,RF,87.8,"24.0 [12.2, 38.9]","2.0 [0.4, 4.6]",52,5,0.959


headline detection main


,Features,ML,Acc (%),FNR % [95% CI],FPR % [95% CI],FN,FP,AUROC
0,Our 7D,RF,98.1,"1.8 [0.4, 3.6]","2.0 [0.4, 4.4]",4,5,0.989
1,Our 7D,GBR,92.7,"2.3 [0.6, 4.1]","11.6 [3.2, 21.7]",5,29,0.983
2,Our 7D,Ridge,81.0,"16.6 [5.8, 29.6]","21.1 [7.7, 39.1]",36,53,0.936
3,|PC1| only,RF,88.7,"18.9 [7.8, 32.1]","4.8 [1.6, 9.2]",41,12,0.939
4,Persona,Ridge,84.6,"3.7 [0.0, 9.9]","25.5 [11.8, 42.0]",8,64,0.939
5,\|Δ \bar{h}\|_2,RF,87.8,"24.0 [12.2, 38.9]","2.0 [0.4, 4.6]",52,5,0.959
6,Training loss,RF,69.7,"47.0 [31.7, 65.8]","15.9 [12.1, 20.1]",102,40,0.763
7,Soligo PCA-7,Ridge,90.6,"18.0 [7.2, 31.3]","2.0 [0.4, 4.1]",39,5,0.957
8,PLS-7,Ridge,89.7,"19.8 [9.3, 32.8]","2.0 [0.4, 4.1]",43,5,0.960
9,"SAE (trained, K=256)",Ridge,74.6,"31.8 [16.9, 49.1]","19.9 [5.5, 36.6]",69,50,0.873


,Table,Matches paper artifact
0,tab_headline_detection.tex,True
1,tab_headline_detection_main.tex,True


In [4]:
state = report["state"]
print(f'Calibration/held-out protocol: {state["CAL_PERTS"]} / {state["OOD_PERTS"]}; threshold = {state["EM_THRESH"]:.0%}')
print(f'Held-out checkpoints: {state["n_total_ood"]}; dangerous: {state["n_dangerous_ood"]}')


Calibration/held-out protocol: ['insecure_code_1k', 'gsm8k_1k', 'jailbroken', 'bad_medical'] / ['number_sequence', 'risky_financial', 'subtle_misinfo']; threshold = 6%
Held-out checkpoints: 468; dangerous: 217
